[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/Pesquisa-Operacional-III-A/blob/main/12_Fun_Geradora.ipynb)

## **Pesquisa Operacional III-A**

**Professor:**
- Diogo Ferreira de Lima Silva (TEP-UFF)

# Funções Geradoras e Introdução ao SimPy

---

## Onde estamos na jornada da simulação

| Aula | Conteúdo |
|------|----------|
| 10 | Distribuições de probabilidade e geração de números aleatórios |
| 11 | Simulação manual em tabelas (motor de eventos discretos) |
| **12** | **Funções geradoras e primeiros passos no SimPy** |
| 13 | Simulação de filas no SimPy |
| 14 | Processos com múltiplas atividades e estatísticas avançadas |

Na aula 11 você viu como o motor de simulação funciona: existe um **relógio**, uma **fila de eventos** e um conjunto de **variáveis de estado**. A cada passo, avançamos o relógio até o próximo evento e atualizamos o estado.

Implementar isso manualmente funciona para exemplos pequenos, mas se torna trabalhoso quando o sistema tem muitos tipos de entidade, recursos disputados e estatísticas complexas.

É aí que entra o **SimPy**: uma biblioteca Python que automatiza esse motor, deixando você focar na **lógica do processo**.

Porém, o SimPy usa um mecanismo de Python chamado **função geradora** para representar processos que avançam no tempo. Entender esse mecanismo é a chave para usar o SimPy com fluência.

---

# Parte 1 — Funções Geradoras em Python

## O problema com funções convencionais

Uma função convencional em Python executa do início ao fim e retorna **um único valor** com `return`. Depois disso, ela perde todo o seu estado interno.

In [ ]:
def contador_convencional():
    conta = 0
    while True:
        return conta   # retorna e encerra — conta nunca chega a 1
        conta += 1

minha_conta = contador_convencional()
print(minha_conta)  # 0
print(minha_conta)  # 0 — estado perdido, sempre retorna 0
print(minha_conta)  # 0

## A solução: `yield`

Quando substituímos `return` por `yield`, a função passa a ser uma **geradora**:

- Na primeira chamada a `next()`, executa até o `yield` e **congela** (pausa) seu estado interno.
- Na próxima chamada a `next()`, **retoma de onde parou**, mantendo todas as variáveis locais.

In [ ]:
def contador_gerador():
    conta = 0
    while True:
        yield conta   # pausa aqui e entrega o valor atual
        conta += 1    # quando retomado, incrementa e volta ao yield

minha_conta = contador_gerador()   # cria o objeto gerador (nada executa ainda)

print(next(minha_conta))   # 0 — executa até o primeiro yield
print(next(minha_conta))   # 1 — retoma após o yield, incrementa, pausa no yield novamente
print(next(minha_conta))   # 2
print(next(minha_conta))   # 3

## Outro exemplo: gerando números de uma sequência

In [ ]:
def sequencia_fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = sequencia_fibonacci()
print([next(fib) for _ in range(10)])  # primeiros 10 números de Fibonacci

## Por que isso importa para a simulação?

Pense em um **processo de simulação**:
- Um cliente **chega** → espera por um recurso → é **atendido** → **sai**.

Cada etapa *"espera"* até que algo aconteça (recurso ficar livre, tempo passar). A função geradora modela isso perfeitamente: ela **pausa** em cada espera e **retoma** quando a condição é satisfeita.

O SimPy usa exatamente esse mecanismo internamente. Cada `yield` dentro de um processo SimPy significa *"aguardar até que este evento ocorra"*.

---

# Parte 2 — Primeiros Passos no SimPy

## Os três elementos fundamentais

| Elemento | O que representa | Código |
|----------|-----------------|--------|
| **Environment** | O relógio e o motor de eventos | `env = simpy.Environment()` |
| **timeout** | Passagem de tempo | `yield env.timeout(duração)` |
| **process** | Um processo ativo no ambiente | `env.process(minha_funcao(env))` |

Vamos instalar e importar o SimPy:

In [ ]:
# Descomente a linha abaixo se estiver no Google Colab ou se SimPy não estiver instalado
# !pip install simpy

import simpy
import numpy as np

np.random.seed(10)

## Exemplo 1: Método Pomodoro

Para apresentar o SimPy, vamos usar um exemplo simples: um estudante que alterna ciclos de **trabalho** e **descanso** ao longo de 8 horas.

- Tempo de trabalho: uniforme entre 24 e 26 minutos.
- Tempo de descanso: exponencial com média 5 minutos.

Queremos imprimir na tela cada vez que o estudante inicia trabalho ou descanso.

In [ ]:
def pomodoro(env, nome):
    """Processo de um estudante alternando trabalho e descanso."""
    while True:
        # --- período de trabalho ---
        print(f"{nome} inicia trabalho no tempo {env.now:.1f} min")
        yield env.timeout(np.random.uniform(low=24, high=26))

        # --- período de descanso ---
        print(f"{nome} inicia descanso no tempo {env.now:.1f} min")
        yield env.timeout(np.random.exponential(scale=5))

np.random.seed(10)
env = simpy.Environment()
env.process(pomodoro(env, "Ana"))
env.run(until=480)   # 8 horas = 480 minutos

**Observe:** cada `yield env.timeout(t)` é exatamente o que a simulação manual fazia ao avançar o relógio. O SimPy cuida disso automaticamente.

### Coletando estatísticas

Vamos guardar os tempos de trabalho e descanso para análise:

In [ ]:
tempos_trabalho = []
tempos_descanso = []

def pomodoro_com_stats(env, nome):
    while True:
        t_inicio = env.now
        yield env.timeout(np.random.uniform(low=24, high=26))
        tempos_trabalho.append(env.now - t_inicio)

        t_inicio = env.now
        yield env.timeout(np.random.exponential(scale=5))
        tempos_descanso.append(env.now - t_inicio)

np.random.seed(10)
env = simpy.Environment()
env.process(pomodoro_com_stats(env, "Ana"))
env.run(until=480)

print(f"Ciclos completos: {len(tempos_trabalho)}")
print(f"Tempo médio de trabalho por ciclo: {np.mean(tempos_trabalho):.2f} min")
print(f"Tempo médio de descanso por ciclo: {np.mean(tempos_descanso):.2f} min")

## Exemplo 2: Múltiplos processos simultâneos

O SimPy pode rodar **vários processos em paralelo**. Basta instanciar cada um com `env.process()`.

In [ ]:
np.random.seed(42)

env = simpy.Environment()
env.process(pomodoro(env, "Ana"))
env.process(pomodoro(env, "Bruno"))
env.run(until=60)   # apenas 1 hora para não gerar saída longa

---

# Parte 3 — Recursos: Disputas pelo Mesmo Servidor

O Pomodoro não tinha disputa por recurso. Em sistemas de filas, **clientes competem** pelo mesmo atendente, máquina ou posto.

O SimPy modela isso com `simpy.Resource`:

```python
servidor = simpy.Resource(env, capacity=1)  # 1 atendente

with servidor.request() as req:
    yield req                        # aguarda até o recurso estar livre
    yield env.timeout(tempo_servico) # ocupa o recurso pelo tempo de serviço
# ao sair do 'with', o recurso é liberado automaticamente
```

## Exemplo 3: Loja com um único atendente

Clientes chegam com intervalo exponencial de média 5 min. O atendimento dura entre 3 e 7 min (uniforme). Há apenas **1 atendente**.

Queremos saber o **tempo médio de espera** de cada cliente.

In [ ]:
tempos_espera = []

def cliente(env, nome, atendente):
    chegou = env.now
    with atendente.request() as req:
        yield req                                              # aguarda atendente
        espera = env.now - chegou
        tempos_espera.append(espera)
        yield env.timeout(np.random.uniform(3, 7))            # atendimento

def chegadas(env, atendente):
    i = 1
    while True:
        yield env.timeout(np.random.exponential(5))           # intervalo entre chegadas
        env.process(cliente(env, f"Cliente {i}", atendente))
        i += 1

np.random.seed(42)
env = simpy.Environment()
atendente = simpy.Resource(env, capacity=1)
env.process(chegadas(env, atendente))
env.run(until=480)

print(f"Clientes atendidos       : {len(tempos_espera)}")
print(f"Tempo médio de espera    : {np.mean(tempos_espera):.2f} min")
print(f"Clientes sem espera      : {sum(1 for t in tempos_espera if t == 0)}")

**Conexão com a teoria:** na próxima aula (13) vamos aprofundar a análise de filas no SimPy, comparando os resultados simulados com as fórmulas analíticas (M/M/1, M/M/s).

---

# Exercício — Processo com Caminhos Múltiplos

Um processo fabril possui quatro atividades: **A → (B ou C) → D**.

- 40% dos trabalhos passam por **B** (tempo = 25 min); os demais vão direto para **C**.
- Tempos de processamento: A = 10 min, C = 8 min, D = 5 min.
- As atividades A, B e C compartilham o **Recurso R1** (capacidade 1).
- A atividade D usa o **Recurso R2** (capacidade 1).
- Trabalhos chegam com intervalo fixo de 50 minutos.

**Perguntas:**
1. Qual o **tempo de ciclo esperado** para um trabalho qualquer?
2. Simule 100.000 minutos e calcule o **tempo de ciclo médio observado**.
3. O resultado está próximo do esperado? Execute a simulação várias vezes com diferentes seeds.

*Dica: o tempo de ciclo é o tempo total que um trabalho leva desde a chegada até a saída do processo.*

In [ ]:
import random

# --- Atividades ---
def atividade_A(env, trabalho_id, R1):
    with R1.request() as req:
        yield req
        yield env.timeout(10)

def atividade_B(env, trabalho_id, R1):
    with R1.request() as req:
        yield req
        yield env.timeout(25)

def atividade_C(env, trabalho_id, R1):
    with R1.request() as req:
        yield req
        yield env.timeout(8)

def atividade_D(env, trabalho_id, R2):
    with R2.request() as req:
        yield req
        yield env.timeout(5)

# --- Processo ---
def processo(env, trabalho_id, R1, R2, tempos_ciclo):
    entrada = env.now
    yield env.process(atividade_A(env, trabalho_id, R1))
    if random.random() < 0.4:
        yield env.process(atividade_B(env, trabalho_id, R1))
    yield env.process(atividade_C(env, trabalho_id, R1))
    yield env.process(atividade_D(env, trabalho_id, R2))
    tempos_ciclo.append(env.now - entrada)

# --- Chegadas ---
def chegadas_processo(env, R1, R2, tempos_ciclo):
    i = 1
    while True:
        yield env.timeout(50)
        env.process(processo(env, f"Trabalho {i}", R1, R2, tempos_ciclo))
        i += 1

# --- Simulação ---
random.seed(10)
np.random.seed(10)

tempos_ciclo = []
env = simpy.Environment()
R1 = simpy.Resource(env, capacity=1)
R2 = simpy.Resource(env, capacity=1)
env.process(chegadas_processo(env, R1, R2, tempos_ciclo))
env.run(until=100_000)

print(f"Trabalhos concluídos     : {len(tempos_ciclo)}")
print(f"Tempo de ciclo médio     : {np.mean(tempos_ciclo):.2f} min")
print()
print("Calcule o TC esperado analiticamente e compare!")

In [ ]:
# Rodando 50 replicações para ver a variabilidade
medias_tc = []
for seed in range(50):
    random.seed(seed)
    np.random.seed(seed)
    tc = []
    env = simpy.Environment()
    R1 = simpy.Resource(env, capacity=1)
    R2 = simpy.Resource(env, capacity=1)
    env.process(chegadas_processo(env, R1, R2, tc))
    env.run(until=100_000)
    medias_tc.append(np.mean(tc))

print(f"Média das replicações    : {np.mean(medias_tc):.2f} min")
print(f"Desvio padrão entre reps : {np.std(medias_tc):.2f} min")